# Compare the effect of different dataset initialisations

In [35]:
import json

from pydantic import BaseModel

# locals
from src.configs import get_class_list
from src.preprocess import Instance, WLASLClass
from src.run_types import CentreCropConfig, OG_Sampler
from src.stats import (
    AVAIL_SETS,
    AVAIL_SPLITS,
    HistoGram,
    get_per_instance_stats,
    reverse_preproc_format,
    sort_distribution,
    to_preproc_format,
)
from src.utils import load_rgb_frames_from_video, plt_display_grid
from src.video_dataset import (
    get_labels_path,
    get_transform,
    get_video_path,
    get_wlasl_info,
    load_data_from_json,
)
from src.visualise import plot_distribution


Set some parameters

In [36]:
verbosity = 1
save_files = False
def printv(*args, level=1, **kwargs):
    if level <= verbosity:
        print(*args, **kwargs)
        
split_idx = 3 #change for different split
set_idx = 0 #change for different set
split_options: list[AVAIL_SPLITS] = ["asl100", "asl300", "asl1000", "asl2000"]
set_options: list[AVAIL_SETS] = ['train', 'test', 'val']
split_name: AVAIL_SPLITS = split_options[split_idx]
set_name: AVAIL_SETS = set_options[set_idx]
classes = get_class_list()
metric = 'num_instances'

## load data

In [37]:
printv(f'Selected: {split_name}')
printv(f'Num classes: {len(classes)}', level=2)

all_sets = {}
tot = 0
for set_name in set_options:
    set_path_info = get_wlasl_info(split_name, set_name)
    set_path = get_labels_path(set_name, set_path_info['labels'], set_path_info['label_suff'])
    all_sets[set_name] = reverse_preproc_format(
        load_data_from_json(set_path, policy="strict"),
        classes)
    printv(f'Num classes of {set_name}: {len(all_sets[set_name])}', level=2)
    tot += len(all_sets[set_name])

printv(all_sets[set_name][0]['gloss'], level=2)
printv(all_sets[set_name][0]['instances'][0], level=2)


Selected: asl2000


## get stats

In [38]:
all_stats = {}
for key, item in all_sets.items():
    all_stats[key] = get_per_instance_stats([WLASLClass.model_validate(i) for i in item])
  
printv(all_stats[set_name][classes[-1]][metric],level=2)
printv(all_stats[set_name][classes[-1]].keys())

dict_keys(['num_instances', 'length_distribution', 'signers_distribution', 'source_distribution', 'url_distribution', 'variation_distribution'])


In [39]:
printv(all_stats[set_name][classes[-1]]['num_instances'])
printv(all_stats[set_name][classes[-1]]['length_distribution'])

1
{32: 1}
